In [1]:
import os
from pathlib import Path
from src.utils import *
from src.config import PROCESSED_IMG_DIR, RAW_IMG_DIR
from src.feature_extraction import *
from src.region_visualization import collect_all_rois, build_mosaic, show_mosaic
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import f_classif
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from src.config import ROOT_DIR
import time
import cv2
import random
from multiprocessing import Pool, cpu_count
from functools import partial
from src.feature_extraction import (
    process_one_image_oriented, extract_oriented_rois,
    label_oriented_rois, crop_oriented_roi,
    build_gabor_bank, build_oriented_feature_column_names,
    _order_skeleton_points_fast
)

In [2]:
paths = get_all_image_paths(directory = PROCESSED_IMG_DIR)
separated_paths = seprate_processed_files(paths) # Separate all paths that end in .png
processed_paths = separated_paths[0] # These are the processed images (paths only)
masks_paths = separated_paths[1]     # These are the binary masks (paths only)
skeletons_paths = separated_paths[2] # These are the skeletons (paths only)
xml_paths = []                       # And this for the .xml (paths only)
ground_truth_bboxes = []             # And this for the ground truth bboxes
for img_path in processed_paths:
    xml_path = get_xml_path(img_path)
    xml_path = xml_path.replace('_processed.xml', '_bbox.xml') # Fix the ending from _processed.xml to _bbox.xml
    xml_paths.append(xml_path)
    bbox = parse_stenosis_xml(xml_path)
    ground_truth_bboxes.append(bbox)
print('XML files: ', len(xml_paths))
print('Ground truth bounding boxes: ', len(ground_truth_bboxes))

Total images found: 24975
Processed images: 8325, Masks: 8325, Skeletons: 8325
XML files:  8325
Ground truth bounding boxes:  8325


In [3]:
# ─────────────────────────────────────────────────────────────
# CELL 0b: Build the 1498-image subset (7 frames × 214 series)
# Strategy: for each serie of each patient, select the 7 frames
# centred around the middle frame of that serie.
# ─────────────────────────────────────────────────────────────
from collections import defaultdict

FRAMES_PER_SERIE = 7  # min-of-maxs determined from dataset analysis
TOTAL_ROIS  = 100
RECT_WIDTH  = 100
RECT_HEIGHT = 50
N_SIZES        = 6
N_ORIENTATIONS = 12

# ── Step 1: Group processed paths by patient and serie ───────
# Path structure: .../dataset_processed/<patientID>/<serieID>/<frame>_processed.png
patient_series = defaultdict(lambda: defaultdict(list))

# After initialising patient_series, add:
for path in processed_paths:
    p          = Path(path)
    patient_id = p.parts[-3]
    serie_id   = p.parts[-2]
    patient_series[patient_id][serie_id].append(path)

# Sort frames within each serie
for patient_id in patient_series:
    for serie_id in patient_series[patient_id]:
        patient_series[patient_id][serie_id] = sorted(patient_series[patient_id][serie_id])

# ── Step 2: Select 7 centre frames from every serie ──────────
subset_processed_paths_b = []

for patient_id, series in sorted(patient_series.items()):
    for serie_id, frame_paths in sorted(series.items()):
        n    = len(frame_paths)
        mid  = n // 2
        half = FRAMES_PER_SERIE // 2

        start = mid - half
        end   = start + FRAMES_PER_SERIE

        # Clamp to valid range
        start = max(0, start)
        end   = min(n, start + FRAMES_PER_SERIE)
        start = max(0, end - FRAMES_PER_SERIE)  # re-adjust if end was clamped

        selected = frame_paths[start:end]
        subset_processed_paths_b.extend(selected)

print(f"Total series found    : {sum(len(s) for s in patient_series.values())}")
print(f"Total frames selected : {len(subset_processed_paths_b)}  (expected 7 × 214 = 1498)")

# ── Step 3: Derive aligned mask, skeleton and GT bbox lists ──
subset_masks_paths_b         = [get_skeleton_path(p).replace('_skeleton.png', '_mask.png') for p in subset_processed_paths_b]
subset_skeletons_paths_b     = [get_skeleton_path(p) for p in subset_processed_paths_b]
subset_ground_truth_bboxes_b = [parse_stenosis_xml(get_bbox_xml_path(p)) for p in subset_processed_paths_b]

print(f"Masks paths            : {len(subset_masks_paths_b)}")
print(f"Skeleton paths         : {len(subset_skeletons_paths_b)}")
print(f"GT bbox lists          : {len(subset_ground_truth_bboxes_b)}")
print(f"Images with annotations: {sum(1 for b in subset_ground_truth_bboxes_b if len(b) > 0)}")

subset_dataset_b = list(zip(subset_processed_paths_b, subset_masks_paths_b,
                            subset_skeletons_paths_b, subset_ground_truth_bboxes_b))

OUTPUT_DIR_CSV_SUBSET_B = Path(os.path.join(ROOT_DIR, "notebooks/csv_files/oriented_rois"))
OUTPUT_DIR_CSV_SUBSET_B.mkdir(parents=True, exist_ok=True)

Total series found    : 214
Total frames selected : 1498  (expected 7 × 214 = 1498)
Masks paths            : 1498
Skeleton paths         : 1498
GT bbox lists          : 1498
Images with annotations: 1498


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL A: Extract oriented ROI coordinates and features
#         → roi_oriented_coordinates_subset_b.csv
#         → roi_oriented_features_subset_b.csv
# Uses the 1498-image subset built in Cell 0b.
# ─────────────────────────────────────────────────────────────
gabor_bank   = build_gabor_bank(ksize=31, n_orientations=N_ORIENTATIONS, n_sizes=N_SIZES)
feature_cols = build_oriented_feature_column_names(n_filters=len(gabor_bank))

worker_args = [
    (img_path, mask_path, skel_path, gt_boxes,
     gabor_bank, feature_cols, TOTAL_ROIS, RECT_WIDTH, RECT_HEIGHT)
    for img_path, mask_path, skel_path, gt_boxes in subset_dataset_b
]

n_cores = cpu_count()
print(f"Starting parallel oriented ROI extraction over {len(subset_dataset_b)} images "
      f"using {n_cores} cores...")
start = time.time()

with Pool(processes=n_cores) as pool:
    results = pool.map(process_one_image_oriented, worker_args)

rows = [row for image_rows in results for row in image_rows]
elapsed = time.time() - start

df_subset_b = pd.DataFrame(rows)

print(f"\nDone in {elapsed:.1f}s — {len(rows)} ROIs extracted.")
print(f"df shape         : {df_subset_b.shape}")
print(f"Stenosis    (1)  : {(df_subset_b['label'] ==  1).sum()}")
print(f"Undetermined(-1) : {(df_subset_b['label'] == -1).sum()}")
print(f"Healthy     (0)  : {(df_subset_b['label'] ==  0).sum()}")

OUTPUT_DIR_CSV_SUBSET_B.mkdir(parents=True, exist_ok=True)

# ── Save coordinates only ─────────────────────────────────────
coord_cols   = ['roi_name', 'image_name', 'center_x', 'center_y',
                'angle', 'width', 'height', 'label']
coords_path  = OUTPUT_DIR_CSV_SUBSET_B / "roi_oriented_coordinates_subset_1498.csv"
df_subset_b[coord_cols].to_csv(coords_path, index=False)
print(f"Saved coordinates : {coords_path}")

# ── Save full feature matrix ──────────────────────────────────
features_path = OUTPUT_DIR_CSV_SUBSET_B / "roi_oriented_features_subset_1498.csv"
df_subset_b.to_csv(features_path, index=False)
print(f"Saved features    : {features_path}")

Starting coordinate extraction over 1498 images...
Starting image 002_5_0029 
Starting image 002_5_0030 
Starting image 002_5_0031 
Starting image 002_5_0032 
Starting image 002_5_0033 
Starting image 002_5_0034 
Starting image 002_5_0035 
Starting image 002_8_0015 
Starting image 002_8_0016 


In [ ]:
# ─────────────────────────────────────────────────────────────
# SENSITIVITY ANALYSIS
# Only the first 100 ROIs per image are pipeline-proposed.
# The manually-centred GT ROIs appended afterwards must be
# excluded so they don't inflate the sensitivity estimate.
# ─────────────────────────────────────────────────────────────
df_coords = pd.read_csv(OUTPUT_DIR_CSV_SUBSET_B / "roi_oriented_coordinates_subset_1498.csv")
df_pipeline = (
    df_coords
    .groupby('image_name', sort=False)
    .head(TOTAL_ROIS)
    .reset_index(drop=True)
)
pos_per_image = df_pipeline.groupby('image_name')['label'].apply(
    lambda x: (x == 1).sum()
)

n_total_images   = len(pos_per_image)
n_detected       = (pos_per_image >= 1).sum()   # images with ≥1 positive ROI
sensitivity      = n_detected / n_total_images if n_total_images > 0 else 0.0
avg_pos_per_img  = pos_per_image.mean()
avg_pos_detected = pos_per_image[pos_per_image >= 1].mean()  # among detected only

print("\n" + "═" * 52)
print("  PIPELINE SENSITIVITY REPORT")
print("═" * 52)
print(f"  Total annotated images            : {n_total_images}")
print(f"  Images with ≥1 positive ROI       : {n_detected}")
print(f"  Images with 0 positive ROIs       : {n_total_images - n_detected}")
print(f"  Sensitivity                       : {sensitivity:.4f}  ({sensitivity*100:.2f}%)")
print("─" * 52)
print(f"  Avg positive ROIs / image (all)   : {avg_pos_per_img:.3f}")
print(f"  Avg positive ROIs / image (detected only): {avg_pos_detected:.3f}")
print("═" * 52)

## EXPLORATION OF EXTRACTED FEATURES

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Load CSVs and reconstruct feature matrix
# ─────────────────────────────────────────────────────────────
df_features_1498  = pd.read_csv(OUTPUT_DIR_CSV_SUBSET_B / "roi_oriented_features_subset_1498.csv")
meta_cols         = ['roi_name', 'image_name', 'center_x', 'center_y',
                     'angle', 'width', 'height', 'label']
y_1498            = df_features_1498['label'].values
X_1498            = df_features_1498.drop(columns=meta_cols).values
feature_cols_1498 = [c for c in df_features_1498.columns if c not in meta_cols]

scaler_1498   = StandardScaler()
X_scaled_1498 = scaler_1498.fit_transform(X_1498)

print(f"Loaded feature matrix : {X_1498.shape}")
print(f"Stenosis     (1)  : {np.sum(y_1498 ==  1)}")
print(f"Undetermined (-1) : {np.sum(y_1498 == -1)}")
print(f"Healthy      (0)  : {np.sum(y_1498 ==  0)}")
print(f"Class imbalance   : {round(np.sum(y_1498 == 0) / max(np.sum(y_1498 == 1), 1), 2)}× more healthy than stenosis")

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: 3D PCA scatter plot
# ─────────────────────────────────────────────────────────────
%matplotlib tk

pca_3d_1498   = PCA(n_components=3)
X_pca_3d_1498 = pca_3d_1498.fit_transform(X_scaled_1498)
total_variance_1498 = pca_3d_1498.explained_variance_ratio_.sum() * 100

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca_3d_1498[y_1498 == 0, 0], X_pca_3d_1498[y_1498 == 0, 1], X_pca_3d_1498[y_1498 == 0, 2],
           alpha=0.4, label='Healthy (0)',  color='#3498db', s=25)
ax.scatter(X_pca_3d_1498[y_1498 == 1, 0], X_pca_3d_1498[y_1498 == 1, 1], X_pca_3d_1498[y_1498 == 1, 2],
           alpha=0.8, label='Stenosis (1)', color='#e74c3c', marker='x', s=45)

ax.set_title(f"3D PCA — 1498-Image Subset\n(Total Explained Variance: {total_variance_1498:.1f}%)",
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel(f"PC 1 ({pca_3d_1498.explained_variance_ratio_[0]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_ylabel(f"PC 2 ({pca_3d_1498.explained_variance_ratio_[1]*100:.1f}%)", fontsize=10, labelpad=10)
ax.set_zlabel(f"PC 3 ({pca_3d_1498.explained_variance_ratio_[2]*100:.1f}%)", fontsize=10, labelpad=10)
ax.xaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.yaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.zaxis.set_pane_color((0.95, 0.95, 0.95, 1.0))
ax.view_init(elev=25, azim=45)
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# EXPLORATION: Feature ranking and boxplots
# ─────────────────────────────────────────────────────────────
%matplotlib inline

f_values_1498, p_values_1498 = f_classif(X_scaled_1498, y_1498)

n_indeces = 8
top_n_indices_1498 = np.argsort(f_values_1498)[::-1][:n_indeces]

print(f"\n--- TOP {n_indeces} MOST DISCRIMINATIVE FEATURES — 1498-Image Subset ---")
for rank, idx in enumerate(top_n_indices_1498, start=1):
    print(f"Rank {rank}: Feature {idx:4d}  '{feature_cols_1498[idx]}'  — F-score: {f_values_1498[idx]:.2f}  p: {p_values_1498[idx]:.2e}")

labels_1498 = np.where(y_1498 == 0, 'Healthy', 'Stenosis')

fig, axes = plt.subplots(1, n_indeces, figsize=(16, 5))
fig.suptitle(f"Top {n_indeces} Most Discriminative Features — 1498-Image Subset", fontsize=16)

for i, feat_idx in enumerate(top_n_indices_1498):
    ax = axes[i]
    sns.boxplot(
        x=labels_1498,
        y=X_scaled_1498[:, feat_idx],
        hue=labels_1498,
        order=['Healthy', 'Stenosis'],
        hue_order=['Healthy', 'Stenosis'],
        palette={'Healthy': '#3498db', 'Stenosis': '#e74c3c'},
        legend=False,
        ax=ax,
    )
    ax.set_title(feature_cols_1498[feat_idx], fontsize=7, wrap=True)
    ax.set_xlabel("")
    ax.set_ylabel("Standardised Value" if i == 0 else "")

plt.tight_layout()
plt.show()